<a href="https://colab.research.google.com/github/jawad-glitch/uncertainty-aware-drone-nav/blob/main/notebooks/kf_gps_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Constant-velocity Kalman filter on noisy GPS

Does not run the simulator: it loads the saved ground-truth flight (`results/pid_flight.npz`),
adds seeded GPS noise with a dropout zone, and filters it.
State `[x, y, vx, vy]`; GPS measures position only; the update is skipped when GPS drops out.

**Colab:** run the first code cell. **Locally:** skip it (repo and venv already set up) and start at the second.

In [ ]:
!git clone https://github.com/jawad-glitch/uncertainty-aware-drone-nav.git
%cd uncertainty-aware-drone-nav
!pip install -r requirements.txt

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src").is_dir() and (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import sensors, filters

data = dict(np.load(ROOT / "results" / "pid_flight.npz"))
meas = sensors.make_sensors(data, seed=0)
valid = meas["gps_valid"]

# q = velocity process noise. 1e-5 laggy, 1e-4 good, 1e-3 jittery
est, sig = filters.run_cv_kf(meas["gps"], valid, 1/48, q=1e-4)

In [ ]:
t = data["time"]
valid = meas["gps_valid"]
true_xy = data["pos"][:, :2]

err = np.linalg.norm(est[:, :2] - true_xy, axis=1)
gps_err = np.linalg.norm(meas["gps"][:, :2] - true_xy, axis=1)   # NaN where dropped
err_x = est[:, 0] - true_xy[:, 0]

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

# 1) path
ax[0].plot(true_xy[:, 0], true_xy[:, 1], label="true")
ax[0].plot(meas["gps"][:, 0], meas["gps"][:, 1], ".", ms=2, alpha=0.4, label="GPS")
ax[0].plot(est[:, 0], est[:, 1], label="KF estimate")
ax[0].set_aspect("equal")
ax[0].legend()
ax[0].set_title("path")

# 2) error over time
ax[1].plot(t, gps_err, alpha=0.4, label="raw GPS error")
ax[1].plot(t, err, label="KF error")
ax[1].fill_between(t, 0, 1, where=~valid, transform=ax[1].get_xaxis_transform(),
                   color="red", alpha=0.2, label="GPS dropout")
ax[1].set_xlabel("time (s)")
ax[1].set_ylabel("position error (m)")
ax[1].legend()
ax[1].set_title("error vs truth")

# 3) does the filter know how wrong it is?
ax[2].plot(t, err_x, label="x error")
ax[2].fill_between(t, -3 * sig[:, 0], 3 * sig[:, 0], alpha=0.25, label="±3σ (filter's belief)")
ax[2].fill_between(t, 0, 1, where=~valid, transform=ax[2].get_xaxis_transform(),
                   color="red", alpha=0.2)
ax[2].set_xlabel("time (s)")
ax[2].set_ylabel("x error (m)")
ax[2].legend()
ax[2].set_title("error vs filter uncertainty")

plt.tight_layout()
plt.show()

print("KF RMSE (all steps):      ", np.sqrt(np.mean(err**2)))
print("KF RMSE (GPS steps only): ", np.sqrt(np.mean(err[valid]**2)))
print("raw GPS RMSE:             ", np.sqrt(np.nanmean(gps_err**2)))
print("KF RMSE inside dropout:   ", np.sqrt(np.mean(err[~valid]**2)))